### TASK 1

In [37]:
# Step 1
#--------------- GIVEN TO US BY THE INSTRUCTOR -----------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
  
# Read the CSV file.
data = pd.read_csv("CTG.csv", skiprows=1)

# Select the relevant numerical columns.
selected_cols = ['LB', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'ASTV', 'MSTV', 'ALTV',
                 'MLTV', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean',
                 'Median', 'Variance', 'Tendency', 'NSP']
data = data[selected_cols].dropna()

# Shuffle the dataset.
data_shuffled = data.sample(frac=1.0, random_state=1337)

# Split into input part X and output part Y.
X = data_shuffled.drop('NSP', axis=1)

# Map the diagnosis code to a human-readable label.
def to_label(y):
    return [None, 'normal', 'suspect', 'pathologic'][(int(y))]

Y = data_shuffled['NSP'].apply(to_label)

# Partition the data into training and test sets.
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2, random_state=0)


#--------------- GIVEN TO US BY THE INSTRUCTOR -----------------

X.head()



,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency
1563,144.0,4.0,0.0,5.0,0.0,0.0,0.0,40.0,0.8,0.0,...,59.0,112.0,171.0,3.0,0.0,160.0,154.0,157.0,8.0,1.0
1241,115.0,6.0,0.0,7.0,1.0,0.0,0.0,23.0,1.5,0.0,...,57.0,94.0,151.0,2.0,0.0,120.0,121.0,122.0,6.0,0.0
21,128.0,3.0,272.0,2.0,2.0,0.0,1.0,26.0,1.7,0.0,...,141.0,57.0,198.0,9.0,0.0,129.0,125.0,132.0,34.0,0.0
98,148.0,0.0,1.0,0.0,1.0,0.0,0.0,61.0,0.5,39.0,...,31.0,130.0,161.0,2.0,0.0,154.0,152.0,154.0,1.0,1.0
39,115.0,9.0,54.0,5.0,0.0,0.0,0.0,27.0,2.3,0.0,...,129.0,53.0,182.0,7.0,0.0,119.0,120.0,120.0,14.0,0.0


In [38]:
# Step 2
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

clf = DummyClassifier(strategy='most_frequent')
#save score as a variable
score = cross_val_score(clf, Xtrain, Ytrain)

# aggrigate function = avg
avg = np.mean(score)
print(avg)

0.7794117647058824


In [39]:
# Step 3
# sklearn.ensemble.RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
# sklearn.linear_model.Perceptron
from sklearn.linear_model import Perceptron
#sklearn.ensemble.GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier


randomforestclf = RandomForestClassifier(n_estimators=100, random_state=1337)
rf_score = cross_val_score(randomforestclf, Xtrain, Ytrain)

perceptronclf = Perceptron(random_state=1337)
perceptron_score = cross_val_score(perceptronclf, Xtrain, Ytrain)

gradientboostingclf = GradientBoostingClassifier(random_state=1337)
gradientboosting_score = cross_val_score(gradientboostingclf, Xtrain, Ytrain)

print("Random Forest Classifier Score:", np.mean(rf_score))
print("Perceptron Classifier Score:", np.mean(perceptron_score))
print("Gradient Boosting Classifier Score:", np.mean(gradientboosting_score))

Random Forest Classifier Score: 0.9388235294117647
Perceptron Classifier Score: 0.7805882352941176
Gradient Boosting Classifier Score: 0.95


In [40]:
# Step 4
from sklearn.metrics import accuracy_score
# Dummy Classifier
clf.fit(Xtrain, Ytrain)
Yguess = clf.predict(Xtest)
print(accuracy_score(Ytest, Yguess))
# Random Forest Classifier
randomforestclf.fit(Xtrain, Ytrain)
Yguess_rf = randomforestclf.predict(Xtest)
print(accuracy_score(Ytest, Yguess_rf))
# Perceptron Classifier
perceptronclf.fit(Xtrain, Ytrain)
Yguess_perceptron = perceptronclf.predict(Xtest)
print(accuracy_score(Ytest, Yguess_perceptron))
# Gradient Boosting Classifier
gradientboostingclf.fit(Xtrain, Ytrain)
Yguess_gradientboosting = gradientboostingclf.predict(Xtest)
print(accuracy_score(Ytest, Yguess_gradientboosting))


0.7746478873239436
0.9483568075117371
0.823943661971831
0.9530516431924883


In [41]:
from collections import Counter
from sklearn.tree import DecisionTreeClassifier as DecisionTree
from sklearn.base import ClassifierMixin as ClassifierMixin

class TreeClassifier(DecisionTree, ClassifierMixin):

    def __init__(self, max_depth=10, criterion='maj_sum'):
        super().__init__(max_depth)
        self.criterion = criterion
        
    def fit(self, X, Y):
        # For decision tree classifiers, there are some different ways to measure
        # the homogeneity of subsets.
        if self.criterion == 'maj_sum':
            self.criterion_function = majority_sum_scorer
        elif self.criterion == 'info_gain':
            self.criterion_function = info_gain_scorer
        elif self.criterion == 'gini':
            self.criterion_function = gini_scorer
        else:
            raise Exception(f'Unknown criterion: {self.criterion}')
        super().fit(X, Y)
        self.classes_ = sorted(set(Y))

    # Select a default value that is going to be used if we decide to make a leaf.
    # We will select the most common value.
    def get_default_value(self, Y):
        self.class_distribution = Counter(Y)
        return self.class_distribution.most_common(1)[0][0]
    
    # Checks whether a set of output values is homogeneous. In the classification case, 
    # this means that all output values are identical.
    # We assume that we called get_default_value just before, so that we can access
    # the class_distribution attribute. If the class distribution contains just one item,
    # this means that the set is homogeneous.
    def is_homogeneous(self, Y):
        return len(self.class_distribution) == 1
        
    # Finds the best splitting point for a given feature. We'll keep frequency tables (Counters)
    # for the upper and lower parts, and then compute the impurity criterion using these tables.
    # In the end, we return a triple consisting of
    # - the best score we found, according to the criterion we're using
    # - the id of the feature
    # - the threshold for the best split
    def best_split(self, X, Y, feature):

        # Create a list of input-output pairs, where we have sorted
        # in ascending order by the input feature we're considering.
        sorted_indices = np.argsort(X[:, feature])        
        X_sorted = list(X[sorted_indices, feature])
        Y_sorted = list(Y[sorted_indices])

        n = len(Y)

        # The frequency tables corresponding to the parts *before and including*
        # and *after* the current element.
        low_distr = Counter()
        high_distr = Counter(Y)

        # Keep track of the best result we've seen so far.
        max_score = -np.inf
        max_i = None

        # Go through all the positions (excluding the last position).
        for i in range(0, n-1):

            # Input and output at the current position.
            x_i = X_sorted[i]
            y_i = Y_sorted[i]
            
            # Update the frequency tables.
            low_distr[y_i] += 1
            high_distr[y_i] -= 1

            # If the input is equal to the input at the next position, we will
            # not consider a split here.
            #x_next = XY[i+1][0]
            x_next = X_sorted[i+1]
            if x_i == x_next:
                continue

            # Compute the homogeneity criterion for a split at this position.
            score = self.criterion_function(i+1, low_distr, n-i-1, high_distr)

            # If this is the best split, remember it.
            if score > max_score:
                max_score = score
                max_i = i

        # If we didn't find any split (meaning that all inputs are identical), return
        # a dummy value.
        if max_i is None:
            return -np.inf, None, None

        # Otherwise, return the best split we found and its score.
        split_point = 0.5*(X_sorted[max_i] + X_sorted[max_i+1])
        return max_score, feature, split_point


In [42]:
def draw_tree(self, graph, node_counter, names):
            node_counter, low_id = self.low_subtree.draw_tree(graph, node_counter, names)
            node_counter, high_id = self.high_subtree.draw_tree(graph, node_counter, names)
            node_id = str(node_counter)
            fname = f'F{self.feature}' if names is None else names[self.feature]
            lbl = f'{fname} > {self.threshold:.4g}?'
            graph.node(node_id, lbl, shape='box', fillcolor='yellow', style='filled, rounded')
            graph.edge(node_id, low_id, 'False')
            graph.edge(node_id, high_id, 'True')
            return node_counter+1, node_id

### TASK 3

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Read the Excel file using Pandas.
alldata = pd.read_excel('Hemnet_data.xlsx')

# # Convert the timestamp string to an integer representing the year.
alldata['year'] = pd.DatetimeIndex(alldata['Sold Date']).year

# Convert 'yes' to 1 and 'no' to 0
alldata['Balcony'] = alldata['Balcony'].map({'Yes': 1, 'No': 0})
alldata['Patio'] = alldata['Patio'].map({'Yes': 1, 'No': 0})
alldata['Lift'] = alldata['Lift'].map({'Yes': 1, 'No': 0})

# Select the 12 input columns and the output column.
selected_columns = ['Final Price (kr)', 'year',  'Num of Room', 'Living Area (m²)', 'Balcony', 'Patio','Current Floor', 'Total Floor', 'Lift', 'Built Year', 'Fee (kr/month)', 'Operating Fee (kr/year)']
alldata = alldata[selected_columns]
alldata = alldata.dropna()

# Shuffle.
alldata_shuffled = alldata.sample(frac=1.0, random_state=0)

# Separate the input and output columns.
X = alldata_shuffled.drop('Final Price (kr)', axis=1)
# For the output, we'll use the log of the sales price.
Y = alldata_shuffled['Final Price (kr)'].apply(np.log)

#----------------------------------------------------

# I added this in order to Remove 'kr' and convert to numeric because it does not work otherwise (data cleaning)
X = X.replace({'kr': ''}, regex=True)

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

#-----------------------------------------------------

# Split into training and test sets.
Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2, random_state=0)

In [45]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate

dummyreg = DummyRegressor()
dummyreg_scores = cross_validate(dummyreg, Xtrain, Ytrain, scoring='neg_mean_squared_error')
dummyreg_mse = -dummyreg_scores['test_score']

print("MSE scores per fold:", dummyreg_mse)
print("Mean MSE score: ", dummyreg_mse.mean())

MSE scores per fold: [0.35548711 0.35827597 0.31759722 0.34236524 0.35596055]
Mean MSE score:  0.34593721954567236


In [46]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

linearreg = LinearRegression()
linearreg_scores = cross_validate(linearreg, Xtrain, Ytrain, scoring='neg_mean_squared_error')
linearreg_mse = -linearreg_scores['test_score']

print("MSE scores per fold :", linearreg_mse)
print("Mean MSE score :", linearreg_mse.mean())

MSE scores per fold : [0.22193545 0.22457376 0.19632154 0.21664858 0.24272344]
Mean MSE score : 0.2204405551756093


In [47]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate

gbreg = GradientBoostingRegressor(random_state=0)
gbreg_scores = cross_validate(gbreg, Xtrain, Ytrain, scoring='neg_mean_squared_error')
gbreg_mse = -gbreg_scores['test_score']

print("MSE scores per fold:", gbreg_mse)
print("Mean MSE score: ", gbreg_mse.mean())

MSE scores per fold: [0.1575823  0.16516694 0.13771891 0.15679981 0.17229262]
Mean MSE score:  0.15791211729893198


In [48]:
from sklearn.ensemble import RandomForestRegressor

randomforestreg = RandomForestRegressor()
randomforestreg_scores = cross_validate(randomforestreg, Xtrain, Ytrain, scoring='neg_mean_squared_error')
randomforestreg_mse = -randomforestreg_scores['test_score']

print("MSE scores per fold:", randomforestreg_mse)
print("Mean MSE score: ", randomforestreg_mse.mean())


MSE scores per fold: [0.14331205 0.1595043  0.13842641 0.14416753 0.15980727]
Mean MSE score:  0.14904351109031963


In [49]:
from sklearn.neural_network import MLPRegressor

mlpreg = MLPRegressor()
mlpreg_scores = cross_validate(mlpreg, Xtrain, Ytrain, scoring='neg_mean_squared_error')
mlpreg_mse = -mlpreg_scores['test_score']

print("MSE scores per fold:", mlpreg_mse)
print("Mean MSE score: ", mlpreg_mse.mean())

c:\Users\oscar\Desktop\kod\AppML\PA1\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\oscar\Desktop\kod\AppML\PA1\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MSE scores per fold: [58.34860721 14.7112893  20.07231506  8.84261722  7.10928397]
Mean MSE score:  21.81682255012261


In [50]:
from sklearn.metrics import mean_squared_error
regr = RandomForestRegressor()

regr.fit(Xtrain, Ytrain)
mean_squared_error(Ytest, regr.predict(Xtest))

0.13460471200338275

NOTES:

The regression models that I have used are: Linear Regression, Gradient Boost Regressor, Random Forest Regressor and MLPRegressor. The model that had the best score was the Random Forest Regressor with a mean MSE(mean square error) score of: 0.14846913793752306 and when we trained it on the full training set it had an MSE score of: 0.13459761730990344

### TASK 4

In [ ]:
!!!!!!!!REMOVE!!!!!!!!!


from collections import Counter
from sklearn.tree import DecisionTreeClassifier as DecisionTree
from sklearn.base import ClassifierMixin as ClassifierMixin

class TreeClassifier(DecisionTree, ClassifierMixin):

    def __init__(self, max_depth=10, criterion='maj_sum'):
        super().__init__(max_depth)
        self.criterion = criterion
        
    def fit(self, X, Y):
        # For decision tree classifiers, there are some different ways to measure
        # the homogeneity of subsets.
        if self.criterion == 'maj_sum':
            self.criterion_function = majority_sum_scorer
        elif self.criterion == 'info_gain':
            self.criterion_function = info_gain_scorer
        elif self.criterion == 'gini':
            self.criterion_function = gini_scorer
        else:
            raise Exception(f'Unknown criterion: {self.criterion}')
        super().fit(X, Y)
        self.classes_ = sorted(set(Y))

    # Select a default value that is going to be used if we decide to make a leaf.
    # We will select the most common value.
    def get_default_value(self, Y):
        self.class_distribution = Counter(Y)
        return self.class_distribution.most_common(1)[0][0]
    
    # Checks whether a set of output values is homogeneous. In the classification case, 
    # this means that all output values are identical.
    # We assume that we called get_default_value just before, so that we can access
    # the class_distribution attribute. If the class distribution contains just one item,
    # this means that the set is homogeneous.
    def is_homogeneous(self, Y):
        return len(self.class_distribution) == 1
        
    # Finds the best splitting point for a given feature. We'll keep frequency tables (Counters)
    # for the upper and lower parts, and then compute the impurity criterion using these tables.
    # In the end, we return a triple consisting of
    # - the best score we found, according to the criterion we're using
    # - the id of the feature
    # - the threshold for the best split
    def best_split(self, X, Y, feature):

        # Create a list of input-output pairs, where we have sorted
        # in ascending order by the input feature we're considering.
        sorted_indices = np.argsort(X[:, feature])        
        X_sorted = list(X[sorted_indices, feature])
        Y_sorted = list(Y[sorted_indices])

        n = len(Y)

        # The frequency tables corresponding to the parts *before and including*
        # and *after* the current element.
        low_distr = Counter()
        high_distr = Counter(Y)

        # Keep track of the best result we've seen so far.
        max_score = -np.inf
        max_i = None

        # Go through all the positions (excluding the last position).
        for i in range(0, n-1):

            # Input and output at the current position.
            x_i = X_sorted[i]
            y_i = Y_sorted[i]
            
            # Update the frequency tables.
            low_distr[y_i] += 1
            high_distr[y_i] -= 1

            # If the input is equal to the input at the next position, we will
            # not consider a split here.
            #x_next = XY[i+1][0]
            x_next = X_sorted[i+1]
            if x_i == x_next:
                continue

            # Compute the homogeneity criterion for a split at this position.
            score = self.criterion_function(i+1, low_distr, n-i-1, high_distr)

            # If this is the best split, remember it.
            if score > max_score:
                max_score = score
                max_i = i

        # If we didn't find any split (meaning that all inputs are identical), return
        # a dummy value.
        if max_i is None:
            return -np.inf, None, None

        # Otherwise, return the best split we found and its score.
        split_point = 0.5*(X_sorted[max_i] + X_sorted[max_i+1])
        return max_score, feature, split_point


In [ ]:

from collections import Counter
from sklearn.tree import DecisionTreeClassifier as DecisionTree
from sklearn.base import RegressorMixin

class TreeRegressor(DecisionTree, RegressorMixin):
    